# Chapter 6: Logistic Regression in Pure Python, TensorFlow, Keras, and PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaewoenlee/ML/blob/main/ch06_logistic_regression_colab.ipynb)

## Run this notebook in Google Colab

1. Click the **Open in Colab** badge above after this notebook has been uploaded to the `jaewoenlee/ML` GitHub repository.
2. In Colab, choose **Runtime → Run all**.
3. TensorFlow, Keras, PyTorch, NumPy, pandas, and matplotlib are normally preinstalled in Google Colab.
4. Run the cells from top to bottom. The four implementations use the same data, learning rate, initial values, and number of epochs so that their results can be compared directly.
5. If Colab asks whether to trust the notebook, review the code and allow execution.

The first code cell below checks whether the notebook is running in Colab and prints the main library versions.

In [ ]:
# Google Colab environment check
import sys
import numpy as np
import matplotlib
import tensorflow as tf
import torch

try:
    import google.colab
    print("Running in Google Colab")
except ImportError:
    print("Running outside Google Colab")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("Matplotlib:", matplotlib.__version__)

## Problem setup

The original example uses the following data and model.

- Input x: study hours = [2, 4, 6, 8, 10, 12, 14]
- Target y: pass/fail = [0, 0, 0, 1, 1, 1, 1]
- Linear model: z = wx + b
- Probability: p = 1 / (1 + exp(-z))
- Loss function: binary cross entropy
- Training: SGD, learning rate = 0.01, 1000 epochs
- Test: predicted probability of passing after studying for 7 hours

This notebook implements the same problem in four ways.

1. Pure Python: manually compute the sigmoid, loss, gradients, and gradient descent
2. TensorFlow: low-level implementation using `tf.Variable`, `GradientTape`, and SGD
3. Keras: high-level implementation using `Sequential` and `Dense`
4. PyTorch: implementation using `nn.Linear` and `BCEWithLogitsLoss`

For an easier comparison, all four models start from w = 0 and b = 0 and use the same learning rate and number of epochs.

Original notebook:
https://github.com/taehojo/deeplearning_4th/blob/master/colab/ch06-colab.ipynb

## Basic idea

Logistic regression first evaluates the linear expression

z = wx + b

and then applies the sigmoid function to convert it into a probability between 0 and 1.

p = 1 / (1 + exp(-z))

For a target value y, the binary cross-entropy loss is

loss = -[y log(p) + (1-y) log(1-p)]

Because this example has only one input variable x, the model has only two trainable parameters: the weight w and the bias b.

# 1. Pure Python implementation

This version does not use NumPy or a deep-learning framework for training. The sigmoid function, loss, gradients, and gradient-descent updates are calculated directly.

In [ ]:
import math
import matplotlib.pyplot as plt

x_py = [2, 4, 6, 8, 10, 12, 14]
y_py = [0, 0, 0, 1, 1, 1, 1]

def sigmoid_py(z):
    # Numerically stable sigmoid calculation
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)

w_py = 0.0
b_py = 0.0
lr = 0.01
epochs = 1000
n = len(x_py)
losses_py = []

for epoch in range(epochs):
    p = [sigmoid_py(w_py*x + b_py) for x in x_py]

    # Binary cross entropy
    eps = 1e-12
    loss = -sum(
        y*math.log(max(prob, eps)) + (1-y)*math.log(max(1-prob, eps))
        for y, prob in zip(y_py, p)
    ) / n
    losses_py.append(loss)

    # Gradients of BCE with the sigmoid model
    dw = sum((prob-y)*x for x, y, prob in zip(x_py, y_py, p)) / n
    db = sum(prob-y for y, prob in zip(y_py, p)) / n

    w_py -= lr * dw
    b_py -= lr * db

print(f'w = {w_py:.6f}, b = {b_py:.6f}, final loss = {losses_py[-1]:.6f}')

hour = 7
pred_py = sigmoid_py(w_py*hour + b_py)
print(f'Predicted probability of passing after {hour} study hours = {pred_py*100:.1f}%')

In [ ]:
x_line = [i/10 for i in range(0, 161)]
y_line_py = [sigmoid_py(w_py*x + b_py) for x in x_line]

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line, y_line_py, label='Python')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 2. Low-level TensorFlow implementation

This version does not use Keras `Sequential` or `Dense` for the model. It trains the parameters directly with `tf.Variable`, `GradientTape`, and SGD.

In [ ]:
import tensorflow as tf

tf.random.set_seed(0)

x_tf = tf.constant([[2.], [4.], [6.], [8.], [10.], [12.], [14.]], dtype=tf.float32)
y_tf = tf.constant([[0.], [0.], [0.], [1.], [1.], [1.], [1.]], dtype=tf.float32)

w_tf = tf.Variable([[0.0]], dtype=tf.float32)
b_tf = tf.Variable([0.0], dtype=tf.float32)
optimizer_tf = tf.optimizers.SGD(learning_rate=0.01)
losses_tf = []

for epoch in range(1000):
    with tf.GradientTape() as tape:
        logits = tf.matmul(x_tf, w_tf) + b_tf
        loss = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(labels=y_tf, logits=logits)
        )

    gradients = tape.gradient(loss, [w_tf, b_tf])
    optimizer_tf.apply_gradients(zip(gradients, [w_tf, b_tf]))
    losses_tf.append(float(loss.numpy()))

print(f'w = {float(w_tf.numpy()[0,0]):.6f}, b = {float(b_tf.numpy()[0]):.6f}, final loss = {losses_tf[-1]:.6f}')

hour_tf = tf.constant([[7.0]], dtype=tf.float32)
pred_tf = float(tf.sigmoid(tf.matmul(hour_tf, w_tf) + b_tf).numpy()[0,0])
print(f'Predicted probability of passing after 7 study hours = {pred_tf*100:.1f}%')

In [ ]:
x_line_tf = tf.reshape(tf.linspace(0.0, 16.0, 161), (-1, 1))
y_line_tf = tf.sigmoid(tf.matmul(x_line_tf, w_tf) + b_tf).numpy().ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line, y_line_tf, label='TensorFlow')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 3. Keras implementation

This version is closest to the original example. It uses one `Dense` neuron with a sigmoid activation function.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

x_keras = np.array([2, 4, 6, 8, 10, 12, 14], dtype=np.float32).reshape(-1, 1)
y_keras = np.array([0, 0, 0, 1, 1, 1, 1], dtype=np.float32)

model_keras = keras.Sequential([
    keras.Input(shape=(1,)),
    layers.Dense(
        1,
        activation='sigmoid',
        kernel_initializer='zeros',
        bias_initializer='zeros'
    )
])

model_keras.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    loss='binary_crossentropy'
)

history_keras = model_keras.fit(x_keras, y_keras, epochs=1000, verbose=0)

w_keras, b_keras = model_keras.layers[0].get_weights()
print(f'w = {w_keras[0,0]:.6f}, b = {b_keras[0]:.6f}, final loss = {history_keras.history["loss"][-1]:.6f}')

pred_keras = float(model_keras.predict(np.array([[7.0]], dtype=np.float32), verbose=0)[0,0])
print(f'Predicted probability of passing after 7 study hours = {pred_keras*100:.1f}%')

In [ ]:
x_line_keras = np.linspace(0, 16, 161, dtype=np.float32).reshape(-1, 1)
y_line_keras = model_keras.predict(x_line_keras, verbose=0).ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line_keras.ravel(), y_line_keras, label='Keras')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 4. PyTorch implementation

`nn.Linear(1, 1)` computes z = wx + b. `BCEWithLogitsLoss` combines the sigmoid operation and binary cross entropy in a numerically stable form.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

x_torch = torch.tensor([[2.], [4.], [6.], [8.], [10.], [12.], [14.]], dtype=torch.float32)
y_torch = torch.tensor([[0.], [0.], [0.], [1.], [1.], [1.], [1.]], dtype=torch.float32)

model_torch = nn.Linear(1, 1)
with torch.no_grad():
    model_torch.weight.zero_()
    model_torch.bias.zero_()

criterion = nn.BCEWithLogitsLoss()
optimizer_torch = optim.SGD(model_torch.parameters(), lr=0.01)
losses_torch = []

for epoch in range(1000):
    optimizer_torch.zero_grad()
    logits = model_torch(x_torch)
    loss = criterion(logits, y_torch)
    loss.backward()
    optimizer_torch.step()
    losses_torch.append(float(loss.item()))

w_torch = float(model_torch.weight.item())
b_torch = float(model_torch.bias.item())
print(f'w = {w_torch:.6f}, b = {b_torch:.6f}, final loss = {losses_torch[-1]:.6f}')

with torch.no_grad():
    pred_torch = float(torch.sigmoid(model_torch(torch.tensor([[7.0]]))).item())
print(f'Predicted probability of passing after 7 study hours = {pred_torch*100:.1f}%')

In [ ]:
x_line_torch = torch.linspace(0, 16, 161).reshape(-1, 1)
with torch.no_grad():
    y_line_torch = torch.sigmoid(model_torch(x_line_torch)).numpy().ravel()

plt.figure(figsize=(7, 4.5))
plt.scatter(x_py, y_py, label='data')
plt.plot(x_line_torch.numpy().ravel(), y_line_torch, label='PyTorch')
plt.axvline(7, linestyle='--', alpha=0.5)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

# 5. Comparison of the four implementations

Because all four implementations use the same initialization and SGD settings, their values of w, b, loss, and the predicted probability at 7 study hours should be nearly identical. Small differences can occur because of floating-point precision and differences in the internal order of calculations.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Method': ['Pure Python', 'TensorFlow', 'Keras', 'PyTorch'],
    'w': [w_py, float(w_tf.numpy()[0,0]), float(w_keras[0,0]), w_torch],
    'b': [b_py, float(b_tf.numpy()[0]), float(b_keras[0]), b_torch],
    'Final loss': [losses_py[-1], losses_tf[-1], history_keras.history['loss'][-1], losses_torch[-1]],
    'P(pass | 7 h)': [pred_py, pred_tf, pred_keras, pred_torch]
})
comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x_py, y_py, s=70, label='data')
plt.plot(x_line, y_line_py, label='Pure Python')
plt.plot(x_line, y_line_tf, '--', label='TensorFlow')
plt.plot(x_line_keras.ravel(), y_line_keras, ':', label='Keras')
plt.plot(x_line_torch.numpy().ravel(), y_line_torch, '-.', label='PyTorch')
plt.axvline(7, linestyle='--', alpha=0.35)
plt.xlabel('Study hours')
plt.ylabel('Pass probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()

## Comparison of code structure

- Pure Python: explicitly calculates the gradients of w and b, making the mathematics of logistic regression easy to inspect.
- TensorFlow: `GradientTape` computes the gradients automatically, while the model equation is still written explicitly.
- Keras: provides the most compact description of the model, loss function, optimizer, and training loop.
- PyTorch: defines the model and optimizer as objects and explicitly follows the sequence forward → loss → backward → step.

For teaching, running the notebook in the order Pure Python → TensorFlow → Keras → PyTorch makes it easy to compare how the same mathematics is expressed in different frameworks.